In [1]:
import pandas as pd
import io

In [ ]:
# 1. Simulasi output mentah DePlot
raw_text = "Kategori | Pangsa Pasar (%) <0x0A> Produk A | 80.0 <0x0A> Produk B | 45.5 <0x0A> Produk C | O.Kosong"
print("Teks Mentah:")
print(raw_text)
print("-" * 30)

1. Teks Mentah:
Kategori | Pangsa Pasar (%) <0x0A> Produk A | 80.0 <0x0A> Produk B | 45.5 <0x0A> Produk C | O.Kosong
------------------------------


In [ ]:
# 2. Proses Parsing (Mengubah teks menjadi CSV format)
clean_text = raw_text.replace("<0x0A>", "\n").replace(" | ", ",")
df = pd.read_csv(io.StringIO(clean_text), skipinitialspace=True)

# Bersihkan spasi
df.columns = df.columns.str.strip()
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

print("Tabel Sebelum Divalidasi:")
display(df)

2. Tabel Sebelum Divalidasi:


,Kategori,Pangsa Pasar (%)
0,Produk A,80.0
1,Produk B,45.5
2,Produk C,O.Kosong


In [ ]:
# 3. Proses Validasi (Membersihkan huruf nyasar & konversi ke angka)
for col in df.columns[1:]:
    # Hapus simbol khusus
    df_temp = df[col].astype(str).str.replace(r'[$,%]', '', regex=True)
    # Konversi paksa ke angka (huruf "O.Kosong" akan menjadi NaN)
    df[col] = pd.to_numeric(df_temp, errors='coerce')

print("Tabel Setelah Konversi Numerik (Siap dicek logikanya):")
display(df)

3. Tabel Setelah Konversi Numerik (Siap dicek logikanya):


,Kategori,Pangsa Pasar (%)
0,Produk A,80.0
1,Produk B,45.5
2,Produk C,NaN


In [5]:
# 4. Cek Logika Persentase
is_valid = True
reason = "Valid"

for col in df.columns:
    if '%' in col.lower():
        total = df[col].sum()
        if total > 105: # Toleransi 5%
            is_valid = False
            reason = f"Logika persentase salah. Total = {total}% (> 100%)"

print("Hasil Evaluasi Akhir:")
print(f"Status Data: {'DITERIMA' if is_valid else 'DITOLAK'}")
print(f"Alasan: {reason}")

Hasil Evaluasi Akhir:
Status Data: DITOLAK
Alasan: Logika persentase salah. Total = 125.5% (> 100%)
